# 01 · Explore the raw data

Before cleaning anything, this notebook checks what is actually in the files from `data/raw/`:

1. Row counts and column types
2. Missing values (nulls)
3. Hidden missing values: `-1` codes that mean "Unknown"

Run `python src/download.py` first to fetch the files.

In [1]:
from pathlib import Path

import pandas as pd

RAW_DIR = Path("..") / "data" / "raw"
pd.set_option("display.width", 120)

## Load the files

The three PriceCatcher months are combined into one table so every check covers all of them.

In [2]:
cpi = pd.read_parquet(RAW_DIR / "cpi_2d_state.parquet")
lookup_item = pd.read_parquet(RAW_DIR / "lookup_item.parquet")
lookup_premise = pd.read_parquet(RAW_DIR / "lookup_premise.parquet")
mcoicop = pd.read_parquet(RAW_DIR / "mcoicop.parquet")

price_files = sorted(RAW_DIR.glob("pricecatcher_*.parquet"))
prices = pd.concat([pd.read_parquet(f) for f in price_files], ignore_index=True)

tables = {
    "prices": prices,
    "cpi": cpi,
    "lookup_item": lookup_item,
    "lookup_premise": lookup_premise,
    "mcoicop": mcoicop,
}
print("PriceCatcher files:", [f.name for f in price_files])

PriceCatcher files: ['pricecatcher_2026-08.parquet', 'pricecatcher_2026-09.parquet', 'pricecatcher_2026-10.parquet']


## 1. Row counts and column types

In [3]:
pd.DataFrame(
    {name: {"rows": len(df), "columns": df.shape[1]} for name, df in tables.items()}
).T

,rows,columns
prices,5035311,4
cpi,45024,4
lookup_item,797,5
lookup_premise,3932,6
mcoicop,343,7


In [4]:
for name, df in tables.items():
    print(f"--- {name}")
    print(df.dtypes.to_string())
    print()

--- prices
date             object
premise_code      int64
item_code         int64
price           float64

--- cpi
state                  str
date        datetime64[ns]
division               str
index              float64

--- lookup_item
item_code        int64
item               str
unit               str
item_group         str
item_category      str

--- lookup_premise
premise_code    int64
premise           str
address           str
premise_type      str
state             str
district          str

--- mcoicop
digits      int64
division      str
group         str
class         str
subclass      str
desc_en       str
desc_bm       str



## 2. Missing values (nulls)

For each table: how many nulls each column has, and what percentage of rows that is.

In [5]:
def null_summary(df):
    """Null count and percentage for every column."""
    nulls = df.isna().sum()
    return pd.DataFrame({"nulls": nulls, "null_pct": (nulls / len(df) * 100).round(2)})


for name, df in tables.items():
    print(f"--- {name}")
    print(null_summary(df).to_string())
    print()

--- prices
              nulls  null_pct
date              0       0.0
premise_code      0       0.0
item_code         0       0.0
price             0       0.0

--- cpi
          nulls  null_pct
state         0       0.0
date          0       0.0
division      0       0.0
index         0       0.0

--- lookup_item
               nulls  null_pct
item_code          0      0.00
item               1      0.13
unit               1      0.13
item_group         1      0.13
item_category      1      0.13

--- lookup_premise
              nulls  null_pct
premise_code      0      0.00
premise           0      0.00
address           0      0.00
premise_type      1      0.03
state             1      0.03
district          1      0.03

--- mcoicop
          nulls  null_pct
digits        0       0.0
division      0       0.0
group         0       0.0
class         0       0.0
subclass      0       0.0
desc_en       0       0.0
desc_bm       0       0.0



## 3. Hidden missing values: `-1` codes

PriceCatcher's lookups use code `-1` for "Unknown". These rows are not null, so step 2 cannot catch them.

In [6]:
print("Lookup rows with code -1:")
print(lookup_item[lookup_item["item_code"] == -1].to_string())
print()
print(lookup_premise[lookup_premise["premise_code"] == -1].to_string())
print()

unknown = pd.DataFrame({
    "rows": [(prices["item_code"] == -1).sum(), (prices["premise_code"] == -1).sum()],
}, index=["item_code == -1", "premise_code == -1"])
unknown["pct_of_prices"] = (unknown["rows"] / len(prices) * 100).round(3)
unknown

Lookup rows with code -1:
   item_code item unit item_group item_category
0         -1  NaN  NaN        NaN           NaN

   premise_code          premise          address premise_type state district
0            -1  Unknown Premise  Unknown Address          NaN   NaN      NaN



,rows,pct_of_prices
item_code == -1,0,0.0
premise_code == -1,0,0.0


## What this shows

- **No nulls in the main data.** All 5,035,311 PriceCatcher rows and all 45,024 CPI rows are complete.
- **The only lookup nulls are the `-1` "Unknown" rows.** `lookup_item` and `lookup_premise` each have exactly one null per column, and it's their `-1` row.
- **No price row uses a `-1` code**, so the Unknown rows are never referenced and can be dropped without losing any data.
- **`mcoicop` uses `-` instead of null** in `group`, `class` and `subclass` for codes where that level does not apply. Only the 2-digit division rows are needed for CPI, so this does not need cleaning.
- **PriceCatcher `date` is stored as text** (`object`), while CPI's is a real date. It needs converting before date filtering or grouping.

Next: check units, price outliers, the October 1–9 gap and the duplicated `Pasar Basah` premise type.